In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.stats import chi2

## Punti di stazione:

1. 063604: punto noto
2. 1002
3. 1005
4. 1006

modello deterministico: y = A x + b --- baselines Δ_ij = X_j − X_i

osservazioni: y0 = [y01, y02, ..., y0n] --- n = # osservazioni

parametri: x = [x1, x2, ..., xm] --- m = # parametri

modello stocastico: y0 = y + v --- v ~ N(0, Q) --- Q = cov(v)

equazioni di osservazione: y0 − b = A x + v

X_hat = (A^T Q^-1 A)^-1 A^T Q^-1 (y0 − b)

Y_hat = A X_hat + b

v_hat = y0 − Y_hat

sigma_v_hat = (v_hat^T Q^-1 v_hat) / (n − m) --- fattore di varianza a posteriori (atteso ≈ 1)

N = A^T Q^-1 A

Cxx = sigma_v_hat * N^-1

Cyy = sigma_v_hat * A N^-1 A^T

Cvv = sigma_v_hat * (Q − A N^-1 A^T)

### 1. Dati di ingresso

In [ ]:
baseline_path = Path("./output/baselines.csv")
# point names stay text: "063604" would otherwise be read as the integer 63604
baselines = pd.read_csv(baseline_path, dtype={"base": str, "rover": str})
baselines


,session,base,rover,epoch,dx,dy,dz,cov_xx,cov_xy,cov_xz,cov_yy,cov_yz,cov_zz,length,Q,ns,ratio,pos_file
0,S1,1005,1002,2026-10-07 11:22:15,84.7704,-5100.2448,871.7748,0.000007,-3.600000e-07,3.240000e-06,3.600000e-07,4.000000e-08,2.560000e-06,5174.908148,1,11,9.2,baselines/S1_1005_1002.pos
1,S1,1006,1002,2026-10-07 11:22:15,-261.6419,-5224.5125,1234.8482,0.000002,4.000000e-08,1.210000e-06,2.500000e-07,1.600000e-07,1.000000e-06,5374.833711,1,11,4.2,baselines/S1_1006_1002.pos
2,S1,063604,1005,2026-10-07 11:22:15,-356.0602,-2145.9613,758.6197,0.000003,4.900000e-07,1.960000e-06,4.900000e-07,4.900000e-07,2.250000e-06,2303.786582,1,11,3.6,baselines/S1_063604_1005.pos
3,S2,1005,1006,2026-10-07 11:33:15,346.4366,124.2627,-363.0804,0.000001,3.600000e-07,1.000000e-06,3.600000e-07,3.600000e-07,1.210000e-06,516.997982,1,11,6.5,baselines/S2_1005_1006.pos
4,S2,063604,1002,2026-10-07 11:33:15,-271.2789,-7246.2125,1630.3871,0.000001,9.000000e-08,6.400000e-07,9.000000e-08,9.000000e-08,4.900000e-07,7432.317938,1,19,4.3,baselines/S2_063604_1002.pos
5,S2,063604,1006,2026-10-07 11:33:15,-9.6129,-2021.7015,395.5523,0.000002,6.400000e-07,1.960000e-06,4.900000e-07,6.400000e-07,1.960000e-06,2060.056064,1,12,6.8,baselines/S2_063604_1006.pos


In [3]:
# Known point (held fixed): marker coordinates, ECEF in metres
fixed = {"063604": np.array([4428814.9484, 859166.7316, 4493637.0158])}

# Baselines to use: fixed ambiguities, ratio >= 3, at least 6 satellites
good = (baselines["Q"] == 1) & (baselines["ratio"] >= 3) & (baselines["ns"] >= 6)
use = baselines[good].reset_index(drop=True)
print(f"{len(use)} of {len(baselines)} baselines used")

# Unknown points = all the points that are not fixed
points = pd.unique(use[["base", "rover"]].to_numpy().ravel())
unknowns = [p for p in points if p not in fixed]
print("unknown points:", unknowns)

n = 3 * len(use)  # number of observations (3 components per baseline)
m = 3 * len(unknowns)  # number of parameters (X, Y, Z of each unknown point)
print(f"n = {n}, m = {m}, redundancy n - m = {n - m}")

6 of 6 baselines used
unknown points: ['1005', '1002', '1006']
n = 18, m = 9, redundancy n - m = 9


### 2. Osservazioni y0 e matrice di covarianza Q

In [4]:
y0 = use[["dx", "dy", "dz"]].to_numpy().ravel()  # [dx1, dy1, dz1, dx2, ...]

# Stochastic model: the formal RTKLIB covariances are far too optimistic for 10 min of static data, so each baseline gets an a-priori precision that grows with its length:
#     sd = a + b * length      (a in metres, b in ppm)
sigma_h = (5e-3, 1.0)  # east and north: 5 mm + 1 ppm
sigma_v = (10e-3, 2.0)  # up:            10 mm + 2 ppm

# The errors are independent in the local east-north-up frame, so Q is built there  and then rotated to ECEF. One rotation at the fixed point is enough for a small network.
lon0, lat0, _ = Transformer.from_crs(
    "EPSG:4978", "EPSG:4979", always_xy=True
).transform(*fixed["063604"])
phi, lam = np.radians(lat0), np.radians(lon0)
R_enu = np.array(  # ECEF -> east, north, up
    [
        [-np.sin(lam), np.cos(lam), 0],
        [-np.sin(phi) * np.cos(lam), -np.sin(phi) * np.sin(lam), np.cos(phi)],
        [np.cos(phi) * np.cos(lam), np.cos(phi) * np.sin(lam), np.sin(phi)],
    ]
)

Q = np.zeros((n, n))  # block diagonal: one 3x3 block per baseline
for k, row in use.iterrows():
    sd_h = sigma_h[0] + sigma_h[1] * 1e-6 * row.length
    sd_v = sigma_v[0] + sigma_v[1] * 1e-6 * row.length
    Q_enu = np.diag([sd_h**2, sd_h**2, sd_v**2])
    Q[3 * k : 3 * k + 3, 3 * k : 3 * k + 3] = R_enu.T @ Q_enu @ R_enu

Q_inv = np.linalg.inv(Q)  # weight matrix

### 3. Matrice di disegno A e termine noto b

In [5]:
# Model of every baseline:   X_rover - X_base = dxyz
# A has +I (3x3) in the columns of the rover, -I in the columns of the base.
# If the rover or the base is the fixed point, its (known) coordinates are moved
# to the right-hand side, in the vector b.
A = np.zeros((n, m))
b = np.zeros(n)

for k, row in use.iterrows():
    rows = slice(3 * k, 3 * k + 3)
    for point, sign in ((row.rover, +1), (row.base, -1)):
        if point in unknowns:
            j = unknowns.index(point)
            A[rows, 3 * j : 3 * j + 3] = sign * np.eye(3)
        else:
            b[rows] += sign * fixed[point]

print(A)

[[-1. -0. -0.  1.  0.  0.  0.  0.  0.]
 [-0. -1. -0.  0.  1.  0.  0.  0.  0.]
 [-0. -0. -1.  0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0. -1. -0. -0.]
 [ 0.  0.  0.  0.  1.  0. -0. -1. -0.]
 [ 0.  0.  0.  0.  0.  1. -0. -0. -1.]
 [ 1.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.  0.  0.  0.]
 [-1. -0. -0.  0.  0.  0.  1.  0.  0.]
 [-0. -1. -0.  0.  0.  0.  0.  1.  0.]
 [-0. -0. -1.  0.  0.  0.  0.  0.  1.]
 [ 0.  0.  0.  1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  1.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  1.]]


### 4. Soluzione, residui e fattore di varianza

In [6]:
N = A.T @ Q_inv @ A  # normal matrix
X_hat = np.linalg.solve(N, A.T @ Q_inv @ (y0 - b))  # estimated parameters
Y_hat = A @ X_hat + b  # adjusted observations
v_hat = y0 - Y_hat  # residuals

sigma_v_hat = (v_hat @ Q_inv @ v_hat) / (n - m)  # a-posteriori variance factor
print(f"sigma_v_hat = {sigma_v_hat:.2f}   (should be close to 1)")

# Chi-square test: is sigma_v_hat compatible with 1 at 95 %?
lo, hi = chi2.ppf(0.025, n - m) / (n - m), chi2.ppf(0.975, n - m) / (n - m)
print(
    f"95 % interval = [{lo:.2f}, {hi:.2f}] ->",
    "PASS" if lo <= sigma_v_hat <= hi else "FAIL",
)

sigma_v_hat = 0.38   (should be close to 1)
95 % interval = [0.30, 2.11] -> PASS


### 5. Matrici di covarianza e affidabilità

In [7]:
N_inv = np.linalg.inv(N)
Cxx = sigma_v_hat * N_inv  # covariance of the parameters
Cyy = sigma_v_hat * (A @ N_inv @ A.T)  # covariance of the adjusted observations
Cvv = sigma_v_hat * (Q - A @ N_inv @ A.T)  # covariance of the residuals

# Redundancy matrix (its trace is n - m) and normalised residuals
R = (Q - A @ N_inv @ A.T) @ Q_inv
tau = np.abs(v_hat) / np.sqrt(np.diag(Cvv))

print("trace(R) =", round(np.trace(R), 2), "(must be", n - m, ")")
print("largest normalised residual:", round(tau.max(), 2), "(outlier if > 3.29)")

trace(R) = 9.0 (must be 9 )
largest normalised residual: 1.81 (outlier if > 3.29)


### 6. Risultati (ECEF)

In [8]:
sd = np.sqrt(np.diag(Cxx))  # standard deviations of X, Y, Z of each point

coords = pd.DataFrame({
    "point": unknowns,
    "X": X_hat[0::3],
    "Y": X_hat[1::3],
    "Z": X_hat[2::3],
    "sd_X": sd[0::3],
    "sd_Y": sd[1::3],
    "sd_Z": sd[2::3],
}).set_index("point")
print(coords.to_string(float_format="{:.4f}".format))

resid = pd.DataFrame({
    "session": use.session,
    "base": use.base,
    "rover": use.rover,
    "vx_mm": v_hat[0::3] * 1e3,
    "vy_mm": v_hat[1::3] * 1e3,
    "vz_mm": v_hat[2::3] * 1e3,
    "tau": [tau[3 * k : 3 * k + 3].max() for k in range(len(use))],
})
print(resid.to_string(float_format="{:.2f}".format))

                 X           Y            Z   sd_X   sd_Y   sd_Z
point                                                           
1005  4428458.8931 857020.7686 4494395.6393 0.0052 0.0034 0.0052
1002  4428543.6738 851920.5207 4495267.4099 0.0070 0.0046 0.0071
1006  4428805.3295 857145.0312 4494032.5623 0.0051 0.0034 0.0052
  session    base rover  vx_mm  vy_mm  vz_mm  tau
0      S1    1005  1002 -10.32   3.05   4.17 1.41
1      S1    1006  1002  13.75  -2.04   0.56 1.81
2      S1  063604  1005  -4.92   1.72  -3.80 1.02
3      S2    1005  1006   0.22   0.09  -3.39 1.14
4      S2  063604  1002  -4.34  -1.63  -7.03 0.71
5      S2  063604  1006   6.00  -1.09   5.81 1.33


### 7. Coordinate e scarti in UTM 32N

In [9]:
# ECEF (WGS84) -> UTM zone 32N: E, N (m) and ellipsoidal height (m)
to_utm = Transformer.from_crs("EPSG:4978", "EPSG:32632", always_xy=True)


def ecef_to_utm(xyz):
    return np.array(to_utm.transform(*xyz))


rows = []
for j, point in enumerate(unknowns):
    X = X_hat[3 * j : 3 * j + 3]  # adjusted ECEF coordinates
    C = Cxx[3 * j : 3 * j + 3, 3 * j : 3 * j + 3]  # their 3x3 covariance

    east, north, h = ecef_to_utm(X)

    # Error propagation  C_utm = J C J^T,  J = d(E, N, h) / d(X, Y, Z) by central differences
    step = 1.0
    J = np.column_stack([
        (ecef_to_utm(X + step * e) - ecef_to_utm(X - step * e)) / (2 * step)
        for e in np.eye(3)
    ])
    sd_E, sd_N, sd_h = np.sqrt(np.diag(J @ C @ J.T))
    rows.append([point, east, north, h, sd_E, sd_N, sd_h])

for point, X in fixed.items():  # the fixed point has no uncertainty
    east, north, h = ecef_to_utm(X)
    rows.append([point, east, north, h, 0.0, 0.0, 0.0])

utm = pd.DataFrame(rows, columns=["point", "E", "N", "h_ell", "sd_E", "sd_N", "sd_h"])
print(utm.set_index("point").to_string(float_format="{:.4f}".format))
utm.to_csv("baselines/mq_utm.csv", index=False, sep=";", decimal=",")

                 E            N   h_ell   sd_E   sd_N   sd_h
point                                                       
1005   653672.4839 4994710.5239 64.8330 0.0033 0.0033 0.0066
1002   648621.1412 4995831.5539 58.9337 0.0045 0.0045 0.0089
1006   653741.0413 4994198.1568 64.5199 0.0033 0.0033 0.0065
063604 655736.7471 4993688.2581 62.6470 0.0000 0.0000 0.0000


## Test with external func

In [10]:
from gnsskit.baseline import df_to_baselines
from gnsskit.network import adjust_network, coordinate_table

adj = adjust_network(
    df_to_baselines(use),
    fixed,
    stochastic="apriori",
    sigma_h=sigma_h,
    sigma_v=sigma_v,
)
print("max |X_hat - gnsskit| =", np.abs(X_hat - adj.xyz.ravel()).max(), "m")
coordinate_table(adj)


max |X_hat - gnsskit| = 0.0 m


,point,fixed,X,Y,Z,sd_X,sd_Y,sd_Z,lat,lon,h_ell,E,N,sd_E,sd_N,sd_h
0,1005,False,4.428459e+06,857020.768575,4.494396e+06,0.005162,0.003392,0.005228,45.089164,10.952812,64.832963,653672.483929,4.994711e+06,0.003303,0.003303,0.006608
1,1002,False,4.428544e+06,851920.520726,4.495267e+06,0.006978,0.004585,0.007067,45.100329,10.888989,58.933668,648621.141233,4.995832e+06,0.004465,0.004465,0.008932
2,1006,False,4.428805e+06,857145.031187,4.494033e+06,0.005109,0.003357,0.005174,45.084540,10.953525,64.519886,653741.041283,4.994198e+06,0.003269,0.003269,0.006539
3,063604,True,4.428815e+06,859166.731600,4.493637e+06,0.000000,0.000000,0.000000,45.079516,10.978710,62.647014,655736.747050,4.993688e+06,0.000000,0.000000,0.000000


## Test scipy

In [11]:
from scipy.optimize import least_squares


# --- the parameters are a flat vector: [X, Y, Z of point 1, X, Y, Z of point 2, ...] ---
def coordinates(x, point):
    """ECEF coordinates of a point: from the parameter vector, or the known ones."""
    if point in fixed:
        return fixed[point]
    j = unknowns.index(point)
    return x[3 * j : 3 * j + 3]


# --- whitening: Q^-1 = L L^T, so that the weighted sum v^T Q^-1 v = |L^T v|^2 ---
L = np.linalg.cholesky(np.linalg.inv(Q))


def residuals(x):
    """One residual per observation: (model - observation), whitened by the covariance."""
    r = np.concatenate([
        coordinates(x, row.rover)
        - coordinates(x, row.base)
        - np.array([row.dx, row.dy, row.dz])
        for _, row in use.iterrows()
    ])
    return L.T @ r


x0 = np.tile(fixed["063604"], len(unknowns))  # start from the known point
sol = least_squares(
    residuals, x0, method="lm"
)  # Levenberg-Marquardt, numerical Jacobian

# --- statistics from the Jacobian of the whitened residuals: N = J^T J ---
J = sol.jac
sigma_v_hat_sp = 2 * sol.cost / (n - m)  # cost = 0.5 * sum(r^2) = 0.5 * v^T Q^-1 v
Cxx_sp = sigma_v_hat_sp * np.linalg.inv(J.T @ J)

print("evaluations of the residual function:", sol.nfev)
print("max |X_hat - scipy| (m):", np.abs(X_hat - sol.x).max())
print(
    "sigma_v_hat: closed form",
    round(sigma_v_hat, 4),
    " scipy",
    round(sigma_v_hat_sp, 4),
)
print("max |Cxx - Cxx_scipy|:", np.abs(Cxx - Cxx_sp).max())


evaluations of the residual function: 3
max |X_hat - scipy| (m): 9.313225746154785e-10
sigma_v_hat: closed form 0.3829  scipy 0.3829
max |Cxx - Cxx_scipy|: 1.1519648082658485e-19


In [12]:
# --- a robust loss (here a 5 cm error added to one baseline) ---
use_bad = use.copy()
use_bad.loc[1, "dz"] += 0.05


def residuals_bad(x):
    r = np.concatenate([
        coordinates(x, row.rover)
        - coordinates(x, row.base)
        - np.array([row.dx, row.dy, row.dz])
        for _, row in use_bad.iterrows()
    ])
    return L.T @ r


plain = least_squares(residuals_bad, x0, method="lm")
# A robust loss flattens the cost for large residuals, so it must start close to the
# solution: start from the plain least-squares estimate. f_scale is in units of sigma.
robust = least_squares(
    residuals_bad, plain.x, method="trf", loss="soft_l1", f_scale=2.0
)

print("plain least squares:", round(np.abs(plain.x - X_hat).max() * 1e3, 1), "mm")
print("soft_l1 robust loss:", round(np.abs(robust.x - X_hat).max() * 1e3, 1), "mm")


plain least squares: 16.3 mm
soft_l1 robust loss: 14.0 mm
